In [5]:
import pandas as pd
import numpy as np
import os


In [6]:
TRANS_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/raw_data/transactions_2016_2017.csv"
TRAIN_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/raw_data/customer_clv_train.csv"
TEST_PATH = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/raw_data/customer_clv_test.csv"
OUTPUT_DIR = "/workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data"
TRAIN_OUT_PATH = f"{OUTPUT_DIR}/train_features.csv"
TEST_OUT_PATH = f"{OUTPUT_DIR}/test_features.csv"
ALL_OUT_PATH = f"{OUTPUT_DIR}/all_features.csv"


In [7]:
print("Loading data...")
required_paths = [TRANS_PATH, TRAIN_PATH, TEST_PATH]
missing_required = [p for p in required_paths if not os.path.exists(p)]
if missing_required:
    raise FileNotFoundError(f"Missing required files: {missing_required}")

transactions = pd.read_csv(TRANS_PATH, parse_dates=['order_date', 'pack_date'])
train_target = pd.read_csv(TRAIN_PATH)
test_target = pd.read_csv(TEST_PATH)
print("Data loaded successfully.")


Loading data...


/tmp/ipykernel_2451/3938347944.py:7: DtypeWarning: Columns (0: prod_size) have mixed types. Specify dtype option on import or set low_memory=False.
  transactions = pd.read_csv(TRANS_PATH, parse_dates=['order_date', 'pack_date'])


Data loaded successfully.


In [81]:
#The features for each transaction are as follows:
#
#cust_id: unique identifier of the customer placing the order
#order_date: date when the order was placed
#pack_date: date when the order was packed/shipped
#sale_id: unique identifier of the sales transaction (multiple articles can be present in the same transaction)
#sale_discount_applied: monetary value of discount applied to the sale
#sale_revenue: final revenue amount received for this line item after discount
#returned_to_shop_id: identifier of the shop/location where the item was returned (empty if not returned)
#prod_id: unique identifier of the purchased product
#prod_size: shoe size of the product
#prod_web_only: binary flag (1/0) indicating whether the product is sold online only
#prod_season: season or collection code (e.g., W14 = Winter 2014)
#prod_brand: brand name of the product
#prod_title: full commercial product name/title
#prod_color: primary color of the product
#prod_type_1: primary target group or category (e.g., men, women, boys)
#prod_type_2: not included (= "shoes")
#prod_type_3: secondary product category (e.g., sneakers, high shoes)
#prod_type_4: tertiary style classification (e.g., high-top sneakers)
#prod_type_5: additional style descriptor (e.g., boots with velcro, dress boots)
#prod_heel: heel type or heel specification (if known)
#prod_material: main outer material of the shoe (e.g., leather, suede) (if known)
#prod_insole: indicator of specific insole feature (if known)
#prod_print: type of print or pattern (if known)
#prod_comfort_sole: indicator of special comfort sole feature (if known)
#prod_comfort_wear: indicator of enhanced comfort wear feature (if known)
#prod_clasp: type of closing mechanism (e.g., velcro, zipper, lace-up) (if known)
#prod_outlet: indicator how often this product was sold through an outlet channel, higher values indicate that the product appeared more often
#
### NOTES
#REAL_REVENUE = sale_revenue + sale_discount_applied
#GROUPING BY CUST_ID <- MONTHS WHERE THE CUSTOMER MADE PURCHASES
#GROUPING BY CUST_ID <- LENGTH OF RETURNED ITEMS (returned_to_shop_id)
#PROD SIZE <- CATEGORICAL VARIABLE
#PROD_TYPE_1 <- TARGET CATEGORY
#PROD_TYPE_2,3,4,5 <- SPECIFIC PRODUCT CATEGORY (PREFERENCES?)
#
#transactions.groupby('cust_id')['sale_revenue'].sum().sort_values(ascending=False)
#transactions.groupby('cust_id').size().sort_values(ascending=False)
#transactions['prod_type_1'].unique()
##Probably exist some correlation between prod_size and prod_type_1
#transactions.loc[transactions['prod_type_1'] == 'boys', 'prod_size'].value_counts()
#transactions['prod_type_3'].unique() #multinomial
#transactions['prod_type_4'].unique() #multi-label
#transactions['prod_type_5'].unique() #Multinomial
#transactions.columns
transactions['prod_material'].unique() #multi-label
#transactions['prod_insole'].unique() #Binary variable, but many missing values
#transactions['prod_print'].unique() #multi-label
#transactions['prod_comfort_sole'].unique() #multi-label
#transactions['prod_comfort_wear'].unique() #multi-label
#transactions['prod_clasp'].unique() #multi-label
#transactions['prod_outlet'].unique() #Binary variable
#transactions['prod_heel'].unique() #ordenal categorical variable
#transactions['prod_size'].unique()
#print('Hey')


<StringArray>
[                                   'leather',
                                      'suede',
                          'synthetic leather',
                                          nan,
                                    'textile',
                                     'nubuck',
                    'patent leather, leather',
          'synthetic leather, patent leather',
                                  'synthetic',
                                     'rubber',
                             'patent leather',
                      'suede, synthetic wool',
                                       'wool',
                      'suede, patent leather',
                     'patent leather, nubuck',
                    'patent leather, textile',
                            'nubuck, textile',
                             'suede, textile',
                             'suede, leather',
                                     'velour',
                    'synthetic wool, leather',

In [8]:
def build_features(data, transactions): 
    trx = transactions.copy()

    #Remove duplicates
    before = len(trx)
    trx.drop_duplicates(inplace=True)
    print(f"drop_duplicates   : {before - len(trx):,} removed ({len(trx):,} remain)")

    # Date formats
    trx['order_date'] = pd.to_datetime(trx['order_date'], format='%Y-%m-%d')
    trx['pack_date'] = pd.to_datetime(trx['pack_date'], format='%Y-%m-%d')
    trx['delivery_time'] = (trx['pack_date'] - trx['order_date']).dt.days

    # Curstomer aggregations
    customer_features = trx.groupby('cust_id').agg(
    customer_revenue=('sale_revenue','sum'),
    customer_purchases_number=('sale_id','nunique'),
    mean_revenue=('sale_revenue', 'mean'),
    n_products=('prod_id', 'nunique') if 'prod_id' in trx.columns else ('sale_revenue', 'size'),
    ).reset_index()

    # Average delivery time per customer
    delivery = trx.groupby('cust_id')['delivery_time'].mean().reset_index()
    customer_features = customer_features.merge(delivery, on='cust_id', how='left')
   
    ## Total discount per customer
    #discount = trx.groupby('cust_id')['sale_discount_applied'].sum().reset_index(name='total_discount')
    #customer_features = customer_features.merge(discount, on='cust_id', how='left')
    web_only = trx.groupby('cust_id')['prod_web_only'].sum().reset_index(name='web_only_purchases')
    customer_features = customer_features.merge(web_only, on='cust_id', how='left')

    ## Fix product size
    #trx['prod_size'] = trx['prod_size'].str.strip()  # remove leading/trailing spaces
    #trx['prod_size'] = trx['prod_size'].str.upper()  # convert to uppercase for consistency
    #size_map = {'XS': 17, 'S':  36, 'M':  38, 'L':  40,'XL': 42} #map sizes to numeric values
    #trx['prod_size'] = trx['prod_size'].replace(size_map) 
    #trx['prod_size'] = pd.to_numeric(trx['prod_size'], errors='coerce')
#
    #bins = [0, 30, 36, 44, 60]
    #labels = ["kids", "youth", "adult", "large"]
    #trx['size_group'] = pd.cut(trx['prod_size'], bins=bins, labels=labels)
    #size_dummies = trx['size_group'].astype(str).str.get_dummies().add_prefix('size_')
    #size_dummies['cust_id'] = trx['cust_id']
    ## How much of each size group each customer bought
    #size_dummies = size_dummies.groupby('cust_id').sum().reset_index()

    # Alpha size grouping (keeps explicit XS/S/M/L/XL signal)
    #size_alpha = trx['prod_size'].astype(str).str.strip().str.upper()
    #alpha_levels = ['XS', 'S', 'M', 'L', 'XL']
    #size_alpha = np.where(pd.Series(size_alpha).isin(alpha_levels), size_alpha, 'OTHER')
    #size_alpha_dummies = pd.get_dummies(pd.Series(size_alpha, index=trx.index), prefix='size_alpha')
    #size_alpha_dummies['cust_id'] = trx['cust_id']
    #size_alpha_dummies = size_alpha_dummies.groupby('cust_id').sum().reset_index()

    # Recency
    last_purchase = trx.groupby('cust_id')['order_date'].max()
    reference = trx['order_date'].max()
    recency = (reference - last_purchase).dt.days.reset_index(name='recency_days')

    first_purchase = trx.groupby('cust_id')['order_date'].min()
    customer_lifespan_days = (last_purchase - first_purchase).dt.days.reset_index(name='customer_lifespan_days')
    tenure_days = (reference - first_purchase).dt.days.reset_index(name='tenure_days')

    month_activity = (
        trx.assign(order_month=trx['order_date'].dt.to_period('M'))
        .groupby('cust_id')['order_month'].nunique()
        .reset_index(name='active_months')
    )
    # Merge all features
    customer_features = customer_features.merge(recency, on='cust_id', how='left')
    ###
    #gamma = 0.05
    #customer_features["recency_decay"] = np.exp(-gamma * customer_features["recency_days"])
    customer_features = customer_features.merge(customer_lifespan_days, on='cust_id', how='left')
    customer_features = customer_features.merge(tenure_days, on='cust_id', how='left')
    customer_features = customer_features.merge(month_activity, on='cust_id', how='left')
    #customer_features = customer_features.merge(size_dummies, on='cust_id', how='left')
    #customer_features = customer_features.merge(size_alpha_dummies, on='cust_id', how='left')
    #customer_features = customer_features.merge(month_dummies, on='cust_id', how='left')
#    
    ## Multi-label features
    ##
    #multiple_labels = ['prod_type_3','prod_material','prod_print','prod_clasp', 'prod_type_4','prod_type_5','prod_comfort_sole','prod_comfort_wear']
    #multi_features = []
    ## Create multi-label features
    #for col in multiple_labels:
        #cleaned = (
            #trx[col]
            #.fillna('unknown')
            #.astype(str)
            #.str.split(',')
            #.apply(lambda vals: ','.join(v.strip() for v in vals))
        #)
        #dummies = cleaned.str.get_dummies(',').add_prefix(col + '_')
        #dummies['cust_id'] = trx['cust_id']
        #dummies = dummies.groupby('cust_id').sum().reset_index()
        #multi_features.append(dummies)
    ## Merge multi-label features
    #for f in multi_features:
        #customer_features = customer_features.merge(f, on='cust_id', how='left')
#    
    # Multinomial features ,'prod_insole'
    multinomial_labels = ['prod_type_1']
    for col in multinomial_labels:
        dummies = (
            trx[col]
            .fillna('unknown')
            .astype(str)
            .str.strip()
            .str.get_dummies()
            .add_prefix(col + '_')
        )
        dummies['cust_id'] = trx['cust_id']
        dummies = dummies.groupby('cust_id').sum().reset_index()
        customer_features = customer_features.merge(dummies, on='cust_id', how='left')


    top_brands = trx["prod_brand"].value_counts().head(5).index.tolist() if "prod_brand" in trx.columns else []

    if len(top_brands) > 0:
        brand_counts = (
            trx[trx["prod_brand"].isin(top_brands)]
            .groupby(["cust_id", "prod_brand"])
            .size()
            .unstack(fill_value=0)
        )
        brand_counts.columns = [f"brand_{str(c)}" for c in brand_counts.columns]
        customer_features = customer_features.merge(brand_counts.reset_index(), on="cust_id", how="left")


    # Return rate and active-month ratio
    returned = trx.groupby('cust_id')['returned_to_shop_id'].apply(
        lambda x: x.notna().sum()
    ).reset_index(name='returned_items')
    customer_features = customer_features.merge(returned, on='cust_id', how='left')
    customer_features['return_rate'] = customer_features['returned_items'] / customer_features['customer_purchases_number'].replace(0, np.nan)
    tenure_months = (customer_features['tenure_days'] / 30.44).clip(lower=1)
    customer_features['active_month_ratio'] = customer_features['active_months'] / tenure_months


    # Encode heel height
    #heel_map = {'<2.5 cm':0, '2.5-5 cm':1, '5-8 cm':2, '>8 cm':3}
    #trx['prod_heel_encoded'] = trx['prod_heel'].map(heel_map).fillna(-1)
    #heel = trx.groupby('cust_id')['prod_heel_encoded'].mean().reset_index()
    #customer_features = customer_features.merge(heel, on='cust_id', how='left')
    #print("Basic features created.")
    # Merge with original customer list to keep all customers in the dataset
    df = data[['cust_id']].merge(customer_features, on='cust_id', how='left')

    # customers without history
    df = df.fillna(0)
    return df



In [9]:
# Build features once for all customers seen in transactions
all_customers = transactions[["cust_id"]].drop_duplicates().reset_index(drop=True)
all_features = build_features(all_customers, transactions)

# Split by customer lists from CLV files
train_features = train_target[["cust_id"]].merge(all_features, on="cust_id", how="inner")
test_features = test_target[["cust_id"]].merge(all_features, on="cust_id", how="left").fillna(0)

os.makedirs(OUTPUT_DIR, exist_ok=True)
all_features.to_csv(ALL_OUT_PATH, index=False)
train_features.to_csv(TRAIN_OUT_PATH, index=False)
test_features.to_csv(TEST_OUT_PATH, index=False)

print(f"Saved: {TRAIN_OUT_PATH} -> {train_features.shape}")
print(f"Saved: {TEST_OUT_PATH} -> {test_features.shape}")
print(f"Saved: {ALL_OUT_PATH} -> {all_features.shape}")


drop_duplicates   : 1,063 removed (343,149 remain)
Saved: /workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data/train_features.csv -> (116591, 23)
Saved: /workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data/test_features.csv -> (29148, 23)
Saved: /workspaces/Transactions-predictive-modeling-on-tabular-data-/data/processed_data/all_features.csv -> (145739, 23)
